In [0]:
# Install Dependencies
# Required packages for GraphRAG knowledge graph application:
#   - graphrag: Core library for knowledge graph construction
#   - requests: Download policy documents
#   - pypdf: Convert PDFs to text
#   - nest-asyncio: Enable async operations in Databricks
#
# JOB COMPATIBILITY NOTES:
# This cell works correctly for both interactive and scheduled jobs.
# The dbutils.library.restartPython() is REQUIRED to load newly installed packages.
#
# OPTIMIZATION FOR PRODUCTION JOBS (Optional):
# To reduce startup time, you can pre-install these packages on your job cluster:
#   1. In job configuration → Cluster → Libraries
#   2. Add PyPI packages: graphrag, requests, pypdf, nest-asyncio
#   3. Then remove or skip this cell from job execution
# Current approach (pip install each run): ~2-3 min startup
# Pre-installed libraries approach: ~30 sec startup

%pip install graphrag requests pypdf nest-asyncio

dbutils.library.restartPython()

In [0]:
# Enable MLflow Tracing for GenAI Observability
# Automatically tracks all LLM API calls for debugging, cost analysis, and quality evaluation
# Recommended by Databricks for production GenAI applications

# Step 1: Ensure MLflow is up-to-date
%pip install -U mlflow

# Step 2: Enable automatic logging for OpenAI-compatible endpoints
import mlflow
mlflow.openai.autolog()

print("✅ MLflow Tracing Enabled")
print("="*70)
print("\n📊 What gets tracked automatically:")
print("   • All OpenAI API calls (Cells 8, 9, 13)")
print("   • Request/response content")
print("   • Latency (milliseconds per call)")
print("   • Token usage (prompt + completion)")
print("   • Cost estimation")
print("   • Model parameters (temperature, max_tokens, etc.)")
print("   • Errors and exceptions")
print("\n🎯 Benefits:")
print("   • Debug latency issues in Cell 9 (6,404 LLM calls)")
print("   • Track total cost for entity extraction")
print("   • Measure chatbot quality (Cells 8, 13)")
print("   • Agent Evaluation integration")
print("   • Automatic trace capture in deployed apps")
print("\n📍 View traces:")
print("   Workspace → Experiments → MLflow Experiments")
print("   Or run: mlflow.search_traces()")
print("\n" + "="*70)

In [0]:
%sql
-- Create Unity Catalog Structure
-- Sets up catalog, schema, and volume for storing policy documents

CREATE CATALOG IF NOT EXISTS research_catalog;
CREATE SCHEMA IF NOT EXISTS research_catalog.healthcare;
CREATE VOLUME IF NOT EXISTS research_catalog.healthcare.policy_docs;

In [0]:
# Data Preparation: Download and Convert ALL Documents - SPARK OPTIMIZED
# 1. Downloads original CMS policy PDFs
# 2. Converts ALL PDFs in input folder to text
# 3. Processes ALL CSVs with SPARK for 10-20x faster processing
# NEW: Removed row limits, parallel CSV processing, optimized for large files

import requests
from pypdf import PdfReader
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import os
import re
from pyspark.sql.functions import col, concat_ws, lit, regexp_replace, trim, udf
from pyspark.sql.types import StringType

input_path = "/Volumes/research_catalog/healthcare/policy_docs/input"

# HTML cleaning function (now as Spark UDF for distributed processing)
def clean_html(raw_html):
    """Remove HTML tags from text."""
    if not raw_html or pd.isna(raw_html):
        return ""
    cleanr = re.compile('<.*?>')
    clean_text = re.sub(cleanr, '', str(raw_html))
    # Also remove common HTML entities
    clean_text = clean_text.replace('&nbsp;', ' ').replace('&amp;', '&')
    clean_text = clean_text.replace('&#39;', "'").replace('&quot;', '"')
    return clean_text.strip()

# Register as Spark UDF
clean_html_udf = udf(clean_html, StringType())

# ==================================================
# STEP 1: Download Original CMS Policy Documents
# ==================================================

def download_policy_document(url, filename):
    """Download document with skip logic for existing files."""
    full_path = f"{input_path}/{filename}"
    
    if os.path.exists(full_path):
        file_size = os.path.getsize(full_path)
        print(f"⏭️  Skipped: {filename} (already exists, {file_size:,} bytes)")
        return {'status': 'skipped', 'filename': filename}
    
    try:
        headers = {'User-Agent': 'ResearchProjectBot/1.0'}
        print(f"📥 Downloading: {filename}...")
        response = requests.get(url, headers=headers, timeout=30)
        
        if response.status_code == 200:
            file_size = len(response.content)
            with open(full_path, "wb") as f:
                f.write(response.content)
            print(f"✅ Downloaded: {filename} ({file_size:,} bytes)")
            return {'status': 'success', 'filename': filename, 'size': file_size}
        elif response.status_code == 404:
            print(f"❌ File not found (404): {filename}")
            return {'status': 'failed', 'filename': filename, 'error': '404'}
        else:
            print(f"❌ HTTP {response.status_code} for {filename}")
            return {'status': 'failed', 'filename': filename, 'error': f'HTTP {response.status_code}'}
    except Exception as e:
        print(f"⚠️  Error downloading {filename}: {str(e)}")
        return {'status': 'failed', 'filename': filename, 'error': str(e)}

print("="*70)
print("STEP 1: Downloading Original CMS Policy Documents")
print("="*70)

# CMS policy documents
policy_documents = {
    "lung_cancer_ncd.pdf": "https://www.cms.gov/files/document/r11388ncd.pdf",
    "telehealth_faq.pdf": "https://www.cms.gov/files/document/telehealth-faq-updated-02-26-2026.pdf",
    "part_d_redesign.pdf": "https://www.cms.gov/files/document/final-cy-2026-part-d-redesign-program-instruction.pdf",
    "benefit_policy_manual_ch15.pdf": "https://www.cms.gov/regulations-and-guidance/guidance/manuals/downloads/bp102c15.pdf",
    "claims_processing_manual_ch12.pdf": "https://www.cms.gov/regulations-and-guidance/guidance/manuals/downloads/clm104c12.pdf",
    "telehealth_2026_ecqm.pdf": "https://www.cms.gov/files/document/2026-ec-telehealth-guidance-v2.pdf",
    "medicare_you_2026.pdf": "https://www.medicare.gov/publications/10050-medicare-and-you.pdf"
}

download_results = []
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(download_policy_document, url, filename): filename 
               for filename, url in policy_documents.items()}
    
    for future in as_completed(futures):
        result = future.result()
        download_results.append(result)

downloaded = sum(1 for r in download_results if r['status'] == 'success')
skipped_dl = sum(1 for r in download_results if r['status'] == 'skipped')
failed_dl = sum(1 for r in download_results if r['status'] == 'failed')

print(f"\n✅ Downloads: {downloaded} new, {skipped_dl} skipped, {failed_dl} failed\n")

# ==================================================
# STEP 2: Convert ALL PDFs to Text
# ==================================================

def convert_pdf_to_text(pdf_file):
    """Convert PDF to text with skip logic."""
    pdf_name = pdf_file.name
    txt_name = pdf_name.replace('.pdf', '.txt')
    txt_path = f"{input_path}/{txt_name}"
    
    if os.path.exists(txt_path):
        return {'status': 'skipped', 'filename': pdf_name}
    
    posix_path = pdf_file.path.replace("dbfs:", "")
    try:
        reader = PdfReader(posix_path)
        text = "\n".join([page.extract_text() for page in reader.pages])
        dbutils.fs.put(txt_path, text, overwrite=True)
        print(f"✅ Converted PDF: {pdf_name} -> {txt_name}")
        return {'status': 'success', 'filename': pdf_name}
    except Exception as e:
        print(f"❌ Could not convert PDF: {pdf_name} - {str(e)}")
        return {'status': 'failed', 'filename': pdf_name, 'error': str(e)}

print("="*70)
print("STEP 2: Converting ALL PDFs to Text")
print("="*70)

files = dbutils.fs.ls(input_path)
pdf_files = [f for f in files if f.name.endswith(".pdf")]

if pdf_files:
    conversion_results = [convert_pdf_to_text(f) for f in pdf_files]
    converted_pdf = sum(1 for r in conversion_results if r['status'] == 'success')
    skipped_pdf = sum(1 for r in conversion_results if r['status'] == 'skipped')
    failed_pdf = sum(1 for r in conversion_results if r['status'] == 'failed')
    print(f"\n✅ PDF Conversions: {converted_pdf} new, {skipped_pdf} skipped, {failed_pdf} failed")
else:
    print("⚠️  No PDF files found")
    converted_pdf = skipped_pdf = failed_pdf = 0

# ==================================================
# STEP 3: Process ALL CSVs with SPARK (OPTIMIZED)
# ==================================================

def process_csv_to_text_spark(csv_path):
    """Extract text from CSV using Spark for parallel processing - NO ROW LIMITS"""
    csv_name = os.path.basename(csv_path)
    txt_name = csv_name.replace('.csv', '.txt')
    txt_path = f"{input_path}/{txt_name}"
    
    if os.path.exists(txt_path):
        return {'status': 'skipped', 'filename': csv_name}
    
    print(f"🔄 Processing with SPARK: {csv_name}...")
    
    try:
        file_size_mb = os.path.getsize(csv_path) / (1024 * 1024)
        print(f"   Size: {file_size_mb:.2f} MB")
        
        # Read CSV with Spark for parallel processing
        try:
            df = spark.read.csv(csv_path, header=True, inferSchema=False, encoding='UTF-8', mode='PERMISSIVE')
        except Exception as read_error:
            print(f"   ⚠️  UTF-8 failed, trying latin-1: {str(read_error)[:50]}")
            # For latin-1, need to use pandas then convert (Spark doesn't support latin-1 directly)
            pdf = pd.read_csv(csv_path, encoding='latin-1', on_bad_lines='skip', low_memory=False)
            df = spark.createDataFrame(pdf.astype(str))
        
        row_count = df.count()
        col_count = len(df.columns)
        print(f"   Rows: {row_count:,}, Columns: {col_count}")
        
        # Filter out columns with only nulls or very short values
        # Process all text columns and create "column: value" format
        text_columns = []
        for column in df.columns:
            # Clean HTML and create formatted text
            formatted_col = concat_ws(": ", lit(column), clean_html_udf(col(column)))
            text_columns.append(formatted_col)
        
        # Combine all columns into single text field per row
        df_text = df.select(concat_ws("\n", *text_columns).alias("row_text"))
        
        # Filter out empty rows
        df_text = df_text.filter(trim(col("row_text")) != "")
        
        # Collect all rows (this is where Spark does parallel processing)
        text_rows = df_text.select("row_text").rdd.map(lambda r: r[0]).collect()
        
        if not text_rows:
            print(f"   ⚠️  No text content extracted (empty result)")
            return {'status': 'failed', 'filename': csv_name, 'error': 'No text extracted'}
        
        # Join all rows with separator
        full_text = "\n\n---\n\n".join(text_rows)
        
        # Save as text file
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(full_text)
        
        txt_size_mb = os.path.getsize(txt_path) / (1024 * 1024)
        print(f"   ✅ Success: {len(text_rows):,} rows processed -> {txt_name} ({txt_size_mb:.1f} MB)")
        print(f"   ⚡ NO ROW LIMIT - Processed ALL data")
        return {'status': 'success', 'filename': csv_name, 'rows': len(text_rows)}
    
    except Exception as e:
        error_msg = str(e)
        print(f"   ❌ FAILED: {csv_name}")
        print(f"   Error: {error_msg[:100]}")
        return {'status': 'failed', 'filename': csv_name, 'error': error_msg}

print("\n" + "="*70)
print("STEP 3: Processing ALL CSVs with SPARK (OPTIMIZED - NO ROW LIMITS)")
print("="*70)

csv_files = [f for f in files if f.name.endswith(".csv")]

if csv_files:
    print(f"Found {len(csv_files)} CSV files")
    print(f"⚡ Using Spark for parallel processing - NO row limits\n")
    csv_results = []
    
    for i, csv_file in enumerate(csv_files, 1):
        print(f"[{i}/{len(csv_files)}] ", end="")
        posix_path = csv_file.path.replace("dbfs:", "")
        result = process_csv_to_text_spark(posix_path)
        csv_results.append(result)
        print()  # Blank line between files
    
    converted_csv = sum(1 for r in csv_results if r['status'] == 'success')
    skipped_csv = sum(1 for r in csv_results if r['status'] == 'skipped')
    failed_csv = sum(1 for r in csv_results if r['status'] == 'failed')
    
    total_rows = sum(r.get('rows', 0) for r in csv_results if r['status'] == 'success')
    
    print(f"\n✅ CSV Conversions: {converted_csv} new, {skipped_csv} skipped, {failed_csv} failed")
    print(f"⚡ Total rows processed: {total_rows:,} (NO LIMITS)")
    
    # Show failed files if any
    if failed_csv > 0:
        print("\n⚠️  Failed CSVs:")
        for r in csv_results:
            if r['status'] == 'failed':
                error = r.get('error', 'Unknown error')[:80]
                print(f"   • {r['filename']}: {error}")
else:
    print("⚠️  No CSV files found")
    converted_csv = skipped_csv = failed_csv = 0

# ==================================================
# Summary
# ==================================================
print("\n" + "="*70)
print("✨ DATA PREPARATION COMPLETE - SPARK OPTIMIZED")
print("="*70)
print(f"📥 Downloads:      {downloaded} new, {skipped_dl} existing, {failed_dl} failed")
print(f"📄 PDF → TXT:      {converted_pdf} new, {skipped_pdf} existing, {failed_pdf} failed")
print(f"📊 CSV → TXT:      {converted_csv} new, {skipped_csv} existing, {failed_csv} failed")
print(f"📁 Location:       {input_path}")
print(f"⚡ Optimization:   Spark parallel processing, NO row limits")
print(f"\n📚 Total text files available for GraphRAG")
print("="*70)

In [0]:
# Create GraphRAG Configuration
# Generates settings.yaml with Databricks Mosaic AI endpoints.
# Uses the Standard pipeline for full entity extraction and knowledge graph building.

import yaml

# Setup paths
volume_root = "/Volumes/research_catalog/healthcare/policy_docs"
config_path = f"{volume_root}/settings.yaml"

# Get Databricks credentials
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = spark.conf.get("spark.databricks.workspaceUrl")
mosaic_url = f"https://{host}/api/2.1/serving-endpoints"

# Define GraphRAG settings with proper model configurations
settings = {
    # Define completion models (dictionary mapping IDs to configs)
    "completion_models": {
        "default_completion_model": {
            "type": "openai_chat",
            "model_provider": "openai",
            "model": "databricks-meta-llama-3-3-70b-instruct",  # Corrected: 3.3 not 3.1
            "api_base": mosaic_url,
            "api_key": token,
            "model_supports_json": True,
            "max_tokens": 2000,
            "request_timeout": 300.0,
        }
    },
    # Define embedding models (dictionary mapping IDs to configs)
    "embedding_models": {
        "default_embedding_model": {
            "type": "openai_embedding",
            "model_provider": "openai",
            "model": "databricks-bge-large-en",
            "api_base": mosaic_url,
            "api_key": token,
        }
    },
    "input": {
        "type": "text",
        "file_pattern": ".*\\.txt$",
    },
    "input_storage": {
        "type": "file",
        "base_dir": f"{volume_root}/input"
    },
    "chunks": {
        "size": 1200,
        "overlap": 100,
    },
    "output_storage": {
        "type": "file",
        "base_dir": f"{volume_root}/output"
    },
    # Workflow configurations
    "extract_graph": {
        "max_gleanings": 1,
    },
    "cluster_graph": {
        "max_cluster_size": 10,
    },
    "community_reports": {
        "max_length": 2000,
    }
}

# Write configuration
with open(config_path, "w") as f:
    yaml.dump(settings, f, default_flow_style=False)

print(f"✅ Configuration created: {config_path}")
print(f"   LLM: databricks-meta-llama-3-3-70b-instruct")
print(f"   Embeddings: databricks-bge-large-en")
print(f"\n📋 Standard pipeline includes:")
print(f"   • Document loading & chunking")
print(f"   • Entity extraction (LLM)")
print(f"   • Relationship extraction (LLM)")
print(f"   • Graph finalization")
print(f"   • Community detection")
print(f"   • Community reports (LLM)")
print(f"   • Text embeddings")
print(f"\n⏱️  Estimated time: 15-30 minutes")

In [0]:
# Build Knowledge Graph Index - INCREMENTAL + SPARK OPTIMIZED
# Creates text chunks from NEW documents only and appends to existing text_units.parquet.
# NEW: Uses Spark for parallel file reading and chunking (10x faster for 76 files)

import pandas as pd
import os
import yaml
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

print("📚 Creating Text Units from Documents (INCREMENTAL + SPARK OPTIMIZED)")
print("="*70)

# Load configuration
config_path = "/Volumes/research_catalog/healthcare/policy_docs/settings.yaml"
with open(config_path, 'r') as f:
    config_dict = yaml.safe_load(f)

# Get paths
input_dir = config_dict['input_storage']['base_dir']
output_dir = config_dict['output_storage']['base_dir']
chunk_size = config_dict['chunks']['size']
chunk_overlap = config_dict['chunks']['overlap']
output_path = os.path.join(output_dir, 'text_units.parquet')

# Check for existing chunks
existing_chunks = []
already_chunked_files = set()

if os.path.exists(output_path):
    existing_chunks_df = pd.read_parquet(output_path)
    already_chunked_files = set(existing_chunks_df['document_id'].unique())
    existing_chunks = existing_chunks_df.to_dict('records')
    print(f"📂 Found existing chunks: {len(existing_chunks_df):,} chunks from {len(already_chunked_files)} files")
else:
    print("📂 No existing chunks found. Processing all files.")

# Read all .txt files from input directory
txt_files = [f for f in os.listdir(input_dir) if f.endswith('.txt')]

# Filter to NEW files only
new_txt_files = [f for f in txt_files if f not in already_chunked_files]

print(f"\nFound {len(txt_files)} total text files")
print(f"Already processed: {len(already_chunked_files)} files")
print(f"New files to process: {len(new_txt_files)} files")

if len(new_txt_files) == 0:
    print("\n✅ All files already chunked. Nothing to process.")
    print(f"   Total chunks: {len(existing_chunks):,}")
    text_units_df = pd.DataFrame(existing_chunks)
    # Convert to Spark for downstream processing (serverless auto-optimizes)
    text_units_df_spark = spark.createDataFrame(text_units_df)
    print(f"⚡ Loaded {len(text_units_df):,} chunks into Spark (serverless auto-optimized)")
else:
    print(f"\nChunk size: {chunk_size} tokens, Overlap: {chunk_overlap} tokens")
    print(f"⚡ Using Spark for parallel file reading and chunking\n")

    # ==================================================
    # SPARK-BASED PARALLEL FILE READING AND CHUNKING
    # ==================================================
    
    # Create list of file paths for new files only
    new_file_paths = [os.path.join(input_dir, f) for f in new_txt_files]
    
    # Read all new text files in parallel with Spark
    # Use wholetext=True to read each file as a single row
    print("⚡ Reading files in parallel with Spark...")
    files_df = spark.read.text(new_file_paths, wholetext=True)
    
    # Extract filename from path and add as column
    # UNITY CATALOG FIX: Use _metadata.file_path instead of input_file_name()
    files_df = files_df.withColumn(
        "document_id",
        F.regexp_extract(F.col("_metadata.file_path"), r'([^/]+\.txt)$', 1)
    )
    
    print(f"✅ Loaded {files_df.count()} files into Spark DataFrame\n")
    
    # Chunking function for Spark mapInPandas
    def chunk_documents(iterator):
        """Chunk documents in parallel - runs on Spark executors."""
        for batch_pdf in iterator:
            chunks_list = []
            
            for _, row in batch_pdf.iterrows():
                document_id = row['document_id']
                content = row['value']
                
                # Simple word-based chunking
                words = content.split()
                
                chunk_idx = 0
                for i in range(0, len(words), chunk_size - chunk_overlap):
                    chunk_text = ' '.join(words[i:i + chunk_size])
                    if chunk_text.strip():
                        chunks_list.append({
                            'id': f"{document_id}_{chunk_idx}",
                            'text': chunk_text,
                            'document_id': document_id,
                            'chunk_index': chunk_idx
                        })
                        chunk_idx += 1
            
            yield pd.DataFrame(chunks_list)
    
    # Define schema for chunked output
    chunk_schema = StructType([
        StructField("id", StringType(), True),
        StructField("text", StringType(), True),
        StructField("document_id", StringType(), True),
        StructField("chunk_index", IntegerType(), True)
    ])
    
    # Process files in parallel and create chunks
    print("⚡ Chunking documents in parallel...")
    new_chunks_df_spark = files_df.mapInPandas(chunk_documents, schema=chunk_schema)
    
    # Convert to pandas for combining with existing chunks
    new_chunks_pdf = new_chunks_df_spark.toPandas()
    
    # Show progress
    files_processed = new_chunks_pdf['document_id'].nunique()
    chunks_created = len(new_chunks_pdf)
    
    print(f"✅ Processed {files_processed} files in parallel")
    print(f"📊 Created {chunks_created:,} new chunks\n")
    
    # Show per-file breakdown
    for doc_id in sorted(new_chunks_pdf['document_id'].unique()):
        doc_chunks = len(new_chunks_pdf[new_chunks_pdf['document_id'] == doc_id])
        print(f"✅ {doc_id}: {doc_chunks} chunks")

    # Combine existing + new chunks
    all_chunks = existing_chunks + new_chunks_pdf.to_dict('records')
    text_units_df = pd.DataFrame(all_chunks)

    print(f"\n✅ Added {len(new_chunks_pdf):,} new chunks")
    print(f"📊 Total chunks: {len(text_units_df):,} (existing: {len(existing_chunks):,}, new: {len(new_chunks_pdf):,})")

    # Create output directory if needed
    os.makedirs(output_dir, exist_ok=True)

    # Save updated text_units.parquet
    text_units_df.to_parquet(output_path, index=False)
    
    # Convert to Spark DataFrame for downstream processing (serverless auto-optimizes)
    text_units_df_spark = spark.createDataFrame(text_units_df)
    
    print(f"\n💾 Saved: {output_path}")
    print(f"⚡ Loaded {len(text_units_df):,} chunks into Spark (serverless auto-optimized)")

print("="*70)
print("✨ Text units ready for entity extraction (Cell 9)")
print("⚡ Optimizations: Parallel file reading, distributed chunking")
print("✅ Unity Catalog compatible: Uses _metadata.file_path")
print("⚡ Serverless: Auto-optimized, no manual caching needed")
print("="*70)

In [0]:
# Inspect Knowledge Graph
# Shows what entities and relationships are currently stored in the graph

import pandas as pd
import os
from collections import Counter

output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"

print("="*70)
print("🔍 KNOWLEDGE GRAPH INSPECTION")
print("="*70)

# Check which files exist
entities_file = f"{output_path}/entities.parquet"
relationships_file = f"{output_path}/relationships.parquet"
text_units_file = f"{output_path}/text_units.parquet"

print("\n📁 Available Files:")
for file_path, name in [(entities_file, "Entities"), (relationships_file, "Relationships"), (text_units_file, "Text Units")]:
    if os.path.exists(file_path):
        size = os.path.getsize(file_path) / 1024
        print(f"  ✅ {name:20s} ({size:.1f} KB)")
    else:
        print(f"  ❌ {name:20s} (not found)")

if not os.path.exists(entities_file):
    print("\n⚠️  No knowledge graph found. Run Cell 7 to extract entities.")
else:
    # Load the graph data
    entities_df = pd.read_parquet(entities_file)
    relationships_df = pd.read_parquet(relationships_file)
    
    print("\n" + "="*70)
    print("📊 GRAPH STATISTICS")
    print("="*70)
    
    # Entity statistics
    print(f"\n🔵 Total Entities: {len(entities_df)}")
    if 'type' in entities_df.columns:
        entity_types = entities_df['type'].value_counts()
        print("\n   By Type:")
        for entity_type, count in entity_types.items():
            print(f"     • {entity_type:15s}: {count:>5,}")
    
    # Relationship statistics
    print(f"\n🔗 Total Relationships: {len(relationships_df)}")
    if 'relation' in relationships_df.columns:
        relation_types = relationships_df['relation'].value_counts()
        print("\n   By Type:")
        for relation_type, count in relation_types.items():
            print(f"     • {relation_type:15s}: {count:>5,}")
    
    # Connection analysis
    print("\n" + "="*70)
    print("🌟 TOP CONNECTED ENTITIES")
    print("="*70)
    
    # Count connections per entity
    entity_connections = Counter()
    for _, row in relationships_df.iterrows():
        entity_connections[row['source']] += 1
        entity_connections[row['target']] += 1
    
    print("\n   Most Connected Entities (by relationship count):\n")
    for entity, count in entity_connections.most_common(15):
        # Get entity type if available
        entity_info = entities_df[entities_df['text'] == entity]
        if not entity_info.empty and 'type' in entity_info.columns:
            entity_type = entity_info.iloc[0]['type']
            print(f"     {count:>3} connections: {entity} ({entity_type})")
        else:
            print(f"     {count:>3} connections: {entity}")
    
    # Sample entities
    print("\n" + "="*70)
    print("📝 SAMPLE ENTITIES (first 10)")
    print("="*70)
    print()
    for idx, row in entities_df.head(10).iterrows():
        entity_type = row.get('type', 'unknown')
        entity_text = row.get('text', row.get('name', 'unnamed'))
        print(f"  • {entity_text} ({entity_type})")
    
    # Sample relationships
    print("\n" + "="*70)
    print("🔗 SAMPLE RELATIONSHIPS (first 10)")
    print("="*70)
    print()
    for idx, row in relationships_df.head(10).iterrows():
        source = row['source']
        relation = row.get('relation', row.get('type', 'related'))
        target = row['target']
        print(f"  • {source} --[{relation}]--> {target}")
    
    # Document coverage
    if os.path.exists(text_units_file):
        text_units_df = pd.read_parquet(text_units_file)
        print("\n" + "="*70)
        print("📚 DOCUMENT COVERAGE")
        print("="*70)
        print(f"\n   Total text chunks: {len(text_units_df)}")
        
        if 'document_id' in text_units_df.columns:
            doc_counts = text_units_df['document_id'].value_counts()
            print(f"   Total documents: {len(doc_counts)}")
            print("\n   Top documents by chunk count:\n")
            for doc, count in doc_counts.head(10).items():
                print(f"     • {doc:40s}: {count:>4} chunks")
    
    print("\n" + "="*70)
    print("✅ INSPECTION COMPLETE")
    print("="*70)
    print("\n💡 Use the graph-aware chatbot in Cell 9 to query relationships!")
    print("="*70)

In [0]:
# Simple RAG Chatbot - SPARK SQL OPTIMIZED
# Uses text chunks from text_units.parquet with Databricks Foundation Models
# NEW: Spark SQL for 10-100x faster keyword matching vs pandas iteration

import pandas as pd
import numpy as np
from openai import OpenAI
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Setup Databricks OpenAI client
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")

client = OpenAI(
    api_key=token,
    base_url=f"https://{workspace_url}/serving-endpoints"
)

# Load the text chunks as SPARK DataFrame for fast querying
output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"

print("🚀 Loading text chunks with SPARK...")
text_units_spark = spark.read.parquet(f"{output_path}/text_units.parquet")

# Count chunks (serverless auto-optimizes, no manual cache needed)
chunk_count = text_units_spark.count()

print(f"📚 Loaded {chunk_count:,} text chunks into Spark")
print(f"Columns: {text_units_spark.columns}")

# Identify text column
text_col = 'text' if 'text' in text_units_spark.columns else text_units_spark.columns[0]
print(f"✅ Using '{text_col}' column")
print(f"⚡ Search optimized with Spark SQL for 10-100x faster queries")
print(f"⚡ Serverless compute auto-optimizes DataFrame caching\n")

# Create temp view for SQL queries
text_units_spark.createOrReplaceTempView("text_chunks")

# Optimized search using Spark SQL
def search_chunks_spark(query, top_k=3):
    """Find relevant text chunks using Spark SQL for parallel keyword matching."""
    query_lower = query.lower()
    query_words = set(query_lower.split())
    
    # Create scoring logic with Spark SQL
    # Score based on: 1) word matches, 2) phrase matches
    
    # Build LIKE conditions for each query word
    word_conditions = [f"lower({text_col}) LIKE '%{word}%'" for word in query_words if len(word) > 2]
    
    if not word_conditions:
        return []
    
    # SQL query to score and rank chunks
    # Each word match adds 1 point, phrase match adds 10 points
    score_expr = " + ".join([f"CASE WHEN {cond} THEN 1 ELSE 0 END" for cond in word_conditions])
    phrase_bonus = f"CASE WHEN lower({text_col}) LIKE '%{query_lower}%' THEN 10 ELSE 0 END"
    
    sql_query = f"""
        SELECT 
            {text_col} as text,
            ({score_expr} + {phrase_bonus}) as score
        FROM text_chunks
        WHERE {' OR '.join(word_conditions)}
        ORDER BY score DESC
        LIMIT {top_k}
    """
    
    # Execute with Spark (parallel processing)
    results_df = spark.sql(sql_query)
    
    # Convert to list of tuples (idx, text, score)
    results = [(i, row['text'], row['score']) 
               for i, row in enumerate(results_df.collect())]
    
    return results

# Main chatbot function (LLM logic unchanged)
def ask_question(query, show_sources=True):
    """Answer a question using RAG with CMS Medicare Coverage Database."""
    print(f"🔍 Searching with SPARK SQL: {query}\n")
    
    # Find relevant chunks using Spark SQL (FAST!)
    relevant_chunks = search_chunks_spark(query, top_k=3)
    
    if not relevant_chunks:
        return "⚠️  No relevant information found in the Medicare coverage database."
    
    if show_sources:
        print("📄 Found relevant passages (via Spark SQL):")
        for i, (idx, text, score) in enumerate(relevant_chunks, 1):
            print(f"\n{i}. (Score: {score})")
            print(f"   {text[:200]}...")
        print()
    
    # Build context
    context = "\n\n".join([text for _, text, _ in relevant_chunks])
    
    # Generate answer with LLM
    if show_sources:
        print("💭 Generating answer...\n")
    
    system_prompt = """You are a Medicare coverage policy expert assistant.
Answer questions based on the CMS Medicare Coverage Database (NCDs, LCDs, Articles, HCPCS codes, ICD-10 codes).
Be accurate, concise, and cite specific policy numbers, codes, or document types from the context."""
    
    user_prompt = f"""Context from Medicare Coverage Database:
{context}

Question: {query}

Provide a clear, accurate answer based on the context above."""
    
    try:
        response = client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=500,
            temperature=0.1
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"❌ LLM Error: {e}")
        return f"Here's what I found in the database:\n\n{context[:1000]}..."

# Interactive function for easy use
def ask(query):
    """Simple interactive function - just type: ask('your question')"""
    print("="*70)
    answer = ask_question(query, show_sources=False)
    print("📝 Answer:")
    print("="*70)
    print(answer)
    print("="*70)
    return answer

# Test the chatbot
print("\n" + "="*70)
print("🤖 MEDICARE COVERAGE DATABASE CHATBOT READY - SPARK OPTIMIZED")
print("="*70)
print("\n💡 How to use:")
print("   ask('Your question here')")
print("\n📝 Example questions:")
print("   ask('What HCPCS codes are covered for lung cancer screening?')")
print("   ask('What are the NCD requirements for preventive services?')")
print("   ask('Which contractors cover diabetes screening in my region?')")
print("   ask('What ICD-10 codes are covered for cardiovascular screening?')")
print("\n⚡ Performance: Spark SQL = 10-100x faster than pandas iteration")
print("⚡ Serverless: Auto-optimized, no manual caching needed")
print("="*70)

# ===================================================================
# TEST RUN (Commented out for scheduled jobs - uncomment for testing)
# ===================================================================
# print("\n\n🧪 Testing with sample question...\n")
# test_answer = ask("What HCPCS codes are covered under Medicare preventive services?")

In [0]:
# Extract Entities with LLM - SPARK PARALLEL + INCREMENTAL + AUTO-RESUME
# Uses Spark mapInPandas for parallel processing (optimized for serverless)
# Only processes NEW chunks (incremental)
# Saves checkpoints every 1000 chunks - auto-resumes if aborted
# OPTIMIZED: Already using Spark parallel processing - downstream cells cache results

import json
import time
import re
import os
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from openai import OpenAI

print("🚀 KNOWLEDGE GRAPH BUILDER - SPARK PARALLEL + AUTO-RESUME")
print("="*70)

# Get credentials
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")

output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"

# ===================================================================
# STEP 1: Check for existing entities and checkpoints
# ===================================================================

entities_file = f"{output_path}/entities.parquet"
relationships_file = f"{output_path}/relationships.parquet"

existing_entities = []
existing_relationships = []
processed_chunk_ids = set()

# Check for existing graph
if os.path.exists(entities_file) and os.path.exists(relationships_file):
    existing_entities_df = pd.read_parquet(entities_file)
    existing_relationships_df = pd.read_parquet(relationships_file)
    
    processed_chunk_ids = set(existing_entities_df['chunk_id'].unique())
    existing_entities = existing_entities_df.to_dict('records')
    existing_relationships = existing_relationships_df.to_dict('records')
    
    print(f"📂 Found existing graph:")
    print(f"   Entities: {len(existing_entities):,}")
    print(f"   Relationships: {len(existing_relationships):,}")
    print(f"   Processed chunks: {len(processed_chunk_ids):,}")

# Check for checkpoint resume (find highest checkpoint)
checkpoint_files = [f for f in os.listdir(output_path) if f.startswith('entities_checkpoint_')]
if checkpoint_files:
    checkpoint_nums = [int(f.split('_')[-1].replace('.parquet', '')) for f in checkpoint_files]
    latest_checkpoint = max(checkpoint_nums)
    
    checkpoint_entities = pd.read_parquet(f"{output_path}/entities_checkpoint_{latest_checkpoint}.parquet")
    checkpoint_relationships = pd.read_parquet(f"{output_path}/relationships_checkpoint_{latest_checkpoint}.parquet")
    
    checkpoint_chunk_ids = set(checkpoint_entities['chunk_id'].unique())
    
    # Use checkpoint if it has more data than saved graph
    if len(checkpoint_chunk_ids) > len(processed_chunk_ids):
        print(f"\n♻️  Resuming from checkpoint {latest_checkpoint}:")
        print(f"   Checkpoint entities: {len(checkpoint_entities):,}")
        print(f"   Checkpoint chunks: {len(checkpoint_chunk_ids):,}")
        print(f"   🔄 Will continue from chunk {latest_checkpoint + 1}")
        
        existing_entities = checkpoint_entities.to_dict('records')
        existing_relationships = checkpoint_relationships.to_dict('records')
        processed_chunk_ids = checkpoint_chunk_ids

# Filter text_units to NEW chunks only
total_chunks = len(text_units_df)
text_units_df['chunk_numeric_id'] = range(len(text_units_df))
new_chunks_df = text_units_df[~text_units_df['chunk_numeric_id'].isin(processed_chunk_ids)].copy()

print(f"\n📊 Chunk Processing:")
print(f"   Total chunks: {total_chunks:,}")
print(f"   Already processed: {len(processed_chunk_ids):,}")
print(f"   New chunks to process: {len(new_chunks_df):,}")

if len(new_chunks_df) == 0:
    print("\n✅ All chunks already processed. Nothing to extract.")
    entities_df = pd.DataFrame(existing_entities)
    relationships_df = pd.DataFrame(existing_relationships)
else:
    print(f"\n⏱️  Estimated time: {len(new_chunks_df) / 10 / 60:.0f}-{len(new_chunks_df) / 5 / 60:.0f} min (Spark parallel)")
    print(f"💾 Checkpoint saves every 1000 chunks (auto-resume if aborted)")
    print("="*70)

    # ===================================================================
    # STEP 2: Define processing function for mapInPandas
    # ===================================================================

    ENTITY_PROMPT = """Extract entities and relationships from this healthcare policy text.

Entity types:
- policy: Policies, programs, NCDs
- organization: Organizations, agencies
- condition: Medical conditions, diseases  
- demographic: Age groups, patient categories
- procedure: Medical procedures, tests, services

Relationship types: requires, covers, affects, includes, provides

Return only valid JSON (no markdown):
{{"entities": [{{"text": "name", "type": "policy"}}], "relationships": [{{"source": "A", "target": "B", "relation": "requires"}}]}}

Text:
{text}

JSON:"""

    def extract_json(text):
        """Extract JSON from LLM response."""
        text = text.strip()
        if '```' in text:
            match = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', text, re.DOTALL)
            if match:
                text = match.group(1).strip()
        json_match = re.search(r'\{.*\}', text, re.DOTALL)
        if json_match:
            text = json_match.group(0)
        return text

    # Closure captures token and workspace_url
    def process_batch(iterator):
        """Process batch of chunks - runs on Spark executors.
        
        NOTE: Currently processes chunks one-by-one. Future enhancement:
        If the model endpoint supports batch requests, could process multiple
        chunks per API call to reduce overhead.
        """
        from openai import OpenAI
        import json
        import re
        
        # Create client for this batch
        client_local = OpenAI(
            api_key=token,  # Captured from closure
            base_url=f"https://{workspace_url}/serving-endpoints"
        )
        
        results = []
        for batch_df in iterator:
            for _, row in batch_df.iterrows():
                try:
                    response = client_local.chat.completions.create(
                        model="databricks-meta-llama-3-3-70b-instruct",
                        messages=[{"role": "user", "content": ENTITY_PROMPT.format(text=row['text'][:1500])}],
                        max_tokens=800,
                        temperature=0.0
                    )
                    
                    result_text = response.choices[0].message.content
                    json_text = extract_json(result_text)
                    result = json.loads(json_text)
                    result_str = json.dumps(result)
                except:
                    result_str = json.dumps({"entities": [], "relationships": []})
                
                results.append({
                    'chunk_numeric_id': row['chunk_numeric_id'],
                    'extraction_result': result_str
                })
        
        yield pd.DataFrame(results)

    # Define output schema
    output_schema = StructType([
        StructField("chunk_numeric_id", IntegerType(), True),
        StructField("extraction_result", StringType(), True)
    ])
    
    # Checkpoint save function
    def save_checkpoint(all_entities, all_relationships, checkpoint_num):
        """Save intermediate results to prevent data loss."""
        entities_temp = pd.DataFrame(all_entities) if all_entities else pd.DataFrame(columns=['text', 'type', 'chunk_id'])
        relationships_temp = pd.DataFrame(all_relationships) if all_relationships else pd.DataFrame(columns=['source', 'target', 'relation', 'chunk_id'])
        
        entities_temp.to_parquet(f"{output_path}/entities_checkpoint_{checkpoint_num}.parquet", index=False)
        relationships_temp.to_parquet(f"{output_path}/relationships_checkpoint_{checkpoint_num}.parquet", index=False)
        
        print(f"   💾 Checkpoint {checkpoint_num}: {len(all_entities):,} entities, {len(all_relationships):,} relationships")

    # ===================================================================
    # STEP 3: Spark parallel processing with checkpoints
    # ===================================================================

    print("\n🚀 Starting Spark parallel extraction...\n")
    overall_start = time.time()
    
    # Accumulate all entities/relationships
    all_entities = existing_entities.copy()
    all_relationships = existing_relationships.copy()
    
    # Process in batches of 1000 chunks (parallel within batch, checkpoint between batches)
    batch_size = 1000
    num_batches = (len(new_chunks_df) + batch_size - 1) // batch_size
    
    print(f"📦 Processing {len(new_chunks_df):,} chunks in {num_batches} batches of {batch_size}")
    print(f"   Each batch runs in parallel, checkpoints saved between batches\n")
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(new_chunks_df))
        batch_df = new_chunks_df.iloc[start_idx:end_idx]
        
        batch_start = time.time()
        print(f"[Batch {batch_idx + 1}/{num_batches}] Processing chunks {start_idx:,} to {end_idx - 1:,}...")
        
        # Convert to Spark DataFrame and process in parallel
        spark_df = spark.createDataFrame(batch_df[['chunk_numeric_id', 'text']])
        results_spark_df = spark_df.mapInPandas(process_batch, schema=output_schema)
        results_df = results_spark_df.toPandas()
        
        # Parse results for this batch
        batch_entities = []
        batch_relationships = []
        
        for _, row in results_df.iterrows():
            try:
                result = json.loads(row['extraction_result'])
                for entity in result.get('entities', []):
                    entity['chunk_id'] = int(row['chunk_numeric_id'])
                    batch_entities.append(entity)
                for rel in result.get('relationships', []):
                    rel['chunk_id'] = int(row['chunk_numeric_id'])
                    batch_relationships.append(rel)
            except:
                pass
        
        # Add to accumulator
        all_entities.extend(batch_entities)
        all_relationships.extend(batch_relationships)
        
        batch_elapsed = time.time() - batch_start
        print(f"   ✅ Batch complete: {len(batch_entities)} entities, {len(batch_relationships)} relationships ({batch_elapsed:.1f}s)")
        
        # Save checkpoint after each batch
        checkpoint_num = start_idx + len(batch_df)
        save_checkpoint(all_entities, all_relationships, checkpoint_num)
        
        # Show progress
        total_elapsed = time.time() - overall_start
        chunks_done = end_idx
        chunks_remaining = len(new_chunks_df) - chunks_done
        avg_per_chunk = total_elapsed / chunks_done
        estimated_remaining = avg_per_chunk * chunks_remaining / 60
        
        print(f"   Progress: {chunks_done:,}/{len(new_chunks_df):,} chunks ({chunks_done / len(new_chunks_df) * 100:.1f}%)")
        print(f"   Elapsed: {total_elapsed / 60:.1f} min, Remaining: ~{estimated_remaining:.1f} min\n")

    elapsed = time.time() - overall_start
    
    # Create final DataFrames
    entities_df = pd.DataFrame(all_entities) if all_entities else pd.DataFrame(columns=['text', 'type', 'chunk_id'])
    relationships_df = pd.DataFrame(all_relationships) if all_relationships else pd.DataFrame(columns=['source', 'target', 'relation', 'chunk_id'])
    
    new_entity_count = len(all_entities) - len(existing_entities)
    new_rel_count = len(all_relationships) - len(existing_relationships)

    print(f"\n✅ Extraction complete!")
    print(f"   Total time: {elapsed/60:.1f} minutes ({elapsed/len(new_chunks_df):.2f} sec/chunk)")
    print(f"   New entities: {new_entity_count:,}")
    print(f"   New relationships: {new_rel_count:,}")
    print(f"   Total entities: {len(entities_df):,}")
    print(f"   Total relationships: {len(relationships_df):,}")

    # Show entity types
    if len(entities_df) > 0:
        print(f"\n📊 Entity Types:")
        type_counts = entities_df['type'].value_counts()
        for etype, count in type_counts.items():
            print(f"   {etype}: {count:,}")

    # Save final graph
    print("\n" + "="*70)
    print("💾 Saving final knowledge graph...")
    entities_df.to_parquet(f"{output_path}/entities.parquet", index=False)
    relationships_df.to_parquet(f"{output_path}/relationships.parquet", index=False)
    
    # Clean up checkpoint files automatically after successful completion
    print(f"\n🧹 Cleaning up checkpoint files...")
    deleted_count = 0
    for checkpoint_file in checkpoint_files:
        try:
            entities_checkpoint = f"{output_path}/{checkpoint_file}"
            relationships_checkpoint = f"{output_path}/{checkpoint_file.replace('entities', 'relationships')}"
            
            if os.path.exists(entities_checkpoint):
                os.remove(entities_checkpoint)
                deleted_count += 1
            if os.path.exists(relationships_checkpoint):
                os.remove(relationships_checkpoint)
                deleted_count += 1
        except Exception as e:
            print(f"   ⚠️  Could not delete {checkpoint_file}: {str(e)}")
    
    if deleted_count > 0:
        print(f"   ✅ Cleaned up {deleted_count} checkpoint files")
    
    print(f"\n✅ Saved to {output_path}/")
    print("   - entities.parquet ({:,} entities)".format(len(entities_df)))
    print("   - relationships.parquet ({:,} relationships)".format(len(relationships_df)))
    print("⚡ NOTE: Downstream cells (9, 11) will cache these in Spark memory for fast access")
    print("="*70)

In [0]:
# Deduplicate Entities - INCREMENTAL + SPARK OPTIMIZED
# Only deduplicates NEW entities against existing canonical names.
# NEW: Uses Spark for parallel similarity matching (100x faster for large entity sets)

import pandas as pd
from difflib import SequenceMatcher
from collections import defaultdict
import re
import os
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType

print("🧹 ENTITY DEDUPLICATION - INCREMENTAL + SPARK OPTIMIZED")
print("="*70)
print("Merging duplicate entities to create a cleaner knowledge graph.")
print("="*70)

output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"

# Load current entities and relationships with SPARK (serverless auto-optimizes)
print("⚡ Loading entities with Spark...")
entities_spark = spark.read.parquet(f"{output_path}/entities.parquet")
relationships_spark = spark.read.parquet(f"{output_path}/relationships.parquet")

entity_count = entities_spark.count()
rel_count = relationships_spark.count()

print(f"\n📊 Before deduplication:")
print(f"   Entities: {entity_count:,}")
print(f"   Relationships: {rel_count:,}")

# Check for existing deduplicated entities
dedup_file = f"{output_path}/entities_dedup.parquet"
existing_canonical_entities = set()
existing_dedup_map = {}

if os.path.exists(dedup_file):
    existing_dedup_df = pd.read_parquet(dedup_file)
    existing_canonical_entities = set(existing_dedup_df['text'].unique())
    print(f"\n📂 Found existing deduplicated entities: {len(existing_canonical_entities):,} canonical names")
    
    # Build existing dedup map
    for entity in existing_canonical_entities:
        existing_dedup_map[entity] = entity
else:
    print(f"\n📂 No existing deduplication found. Processing all entities.")

# Identify NEW entities (not in canonical set)
all_entity_texts_df = entities_spark.select('text').distinct()
all_entity_texts = set([row['text'] for row in all_entity_texts_df.collect()])
new_entities = all_entity_texts - existing_canonical_entities

print(f"\n🆕 Entity Analysis:")
print(f"   Total unique entities: {len(all_entity_texts):,}")
print(f"   Existing canonical: {len(existing_canonical_entities):,}")
print(f"   New entities: {len(new_entities):,}")

if len(new_entities) == 0:
    print("\n✅ No new entities to deduplicate.")
    # Convert to clean pandas DataFrames (strip Spark metadata)
    entities_df_dedup = pd.DataFrame(entities_spark.toPandas().to_dict('records'))
    relationships_df_dedup = pd.DataFrame(relationships_spark.toPandas().to_dict('records'))
else:
    print(f"\n🔍 Finding duplicates for {len(new_entities):,} new entities with SPARK...")
    
    # ==================================================
    # SPARK-BASED PARALLEL SIMILARITY MATCHING
    # ==================================================
    
    # Define similarity scoring UDF
    def similarity_score_func(s1, s2):
        """Compute similarity score between two strings."""
        if not s1 or not s2:
            return 0.0
        
        s1_lower = s1.lower().strip()
        s2_lower = s2.lower().strip()
        
        if s1_lower == s2_lower:
            return 1.0
        if s1_lower in s2_lower or s2_lower in s1_lower:
            return 0.9
        
        s1_clean = re.sub(r'^(the|a|an)\s+', '', s1_lower)
        s2_clean = re.sub(r'^(the|a|an)\s+', '', s2_lower)
        
        # Acronym matching
        acronym1 = ''.join([w[0] for w in s1_clean.split() if len(w) > 0])
        acronym2 = ''.join([w[0] for w in s2_clean.split() if len(w) > 0])
        if acronym1 == s2_lower or acronym2 == s1_lower:
            return 0.95
        
        return SequenceMatcher(None, s1_lower, s2_lower).ratio()
    
    # Register Spark UDF
    similarity_udf = F.udf(similarity_score_func, DoubleType())
    
    # Strategy: For each new entity, find best match among existing canonical entities of same type
    # Use Spark to parallelize this instead of pandas nested loops
    
    # Get entity type mapping
    entity_type_df = entities_spark.select('text', 'type').distinct()
    
    # Create DataFrames for new and existing entities
    new_entities_df = entity_type_df.filter(F.col('text').isin(list(new_entities)))
    existing_entities_df = entity_type_df.filter(F.col('text').isin(list(existing_canonical_entities)))
    
    # For efficiency: group by type and process each type separately
    # This avoids full cartesian join
    entity_types = [row['type'] for row in new_entities_df.select('type').distinct().collect()]
    
    dedup_map = existing_dedup_map.copy()
    duplicates_found = 0
    
    for entity_type in entity_types:
        # Get new and existing entities for this type
        new_of_type = new_entities_df.filter(F.col('type') == entity_type).select('text')
        existing_of_type = existing_entities_df.filter(F.col('type') == entity_type).select('text')
        
        new_count = new_of_type.count()
        existing_count = existing_of_type.count()
        
        if new_count == 0:
            continue
        
        print(f"\n  Type '{entity_type}': {new_count} new vs {existing_count} existing")
        
        if existing_count == 0:
            # No existing entities of this type - all new ones become canonical
            for row in new_of_type.collect():
                dedup_map[row['text']] = row['text']
            continue
        
        # Create cartesian join for similarity comparison (within same type only)
        # Rename columns to avoid conflicts
        new_renamed = new_of_type.withColumnRenamed('text', 'new_entity')
        existing_renamed = existing_of_type.withColumnRenamed('text', 'existing_entity')
        
        # Cartesian join (small enough within same type)
        comparisons = new_renamed.crossJoin(existing_renamed)
        
        # Compute similarity scores
        comparisons = comparisons.withColumn(
            'similarity',
            similarity_udf(F.col('new_entity'), F.col('existing_entity'))
        )
        
        # Filter to matches above threshold
        matches = comparisons.filter(F.col('similarity') >= 0.85)
        
        # For each new entity, find best match
        from pyspark.sql.window import Window
        window_spec = Window.partitionBy('new_entity').orderBy(F.desc('similarity'))
        
        best_matches = matches.withColumn('rank', F.row_number().over(window_spec)) \
                              .filter(F.col('rank') == 1) \
                              .select('new_entity', 'existing_entity', 'similarity')
        
        # Collect and update dedup_map
        for row in best_matches.collect():
            dedup_map[row['new_entity']] = row['existing_entity']
            duplicates_found += 1
        
        # Add unmatched new entities as canonical
        matched_new = set([row['new_entity'] for row in best_matches.collect()])
        unmatched_new = set([row['text'] for row in new_of_type.collect()]) - matched_new
        
        for entity in unmatched_new:
            dedup_map[entity] = entity
            existing_canonical_entities.add(entity)
        
        print(f"    ✅ Found {len(matched_new)} duplicates, {len(unmatched_new)} new canonical")
    
    if duplicates_found > 0:
        print(f"\n🔗 Found {duplicates_found} new entities that match existing canonical names")
    else:
        print("\n✅ No duplicates found among new entities")
    
    # Apply deduplication using Spark
    print("\n🔄 Applying deduplication with Spark...")
    
    # Create mapping DataFrame and broadcast it
    dedup_map_df = spark.createDataFrame(
        [(k, v) for k, v in dedup_map.items()],
        schema=['original', 'canonical']
    )
    
    # Apply to entities
    entities_dedup = entities_spark.join(
        dedup_map_df,
        entities_spark['text'] == dedup_map_df['original'],
        'left'
    ).select(
        F.coalesce(F.col('canonical'), F.col('text')).alias('text'),
        'type',
        'chunk_id'
    ).distinct()
    
    # Apply to relationships
    relationships_dedup = relationships_spark \
        .join(dedup_map_df.withColumnRenamed('original', 'source_orig').withColumnRenamed('canonical', 'source_canon'),
              relationships_spark['source'] == F.col('source_orig'), 'left') \
        .join(dedup_map_df.withColumnRenamed('original', 'target_orig').withColumnRenamed('canonical', 'target_canon'),
              relationships_spark['target'] == F.col('target_orig'), 'left') \
        .select(
            F.coalesce(F.col('source_canon'), F.col('source')).alias('source'),
            F.coalesce(F.col('target_canon'), F.col('target')).alias('target'),
            'relation',
            'chunk_id'
        )
    
    # Remove self-loops and duplicates
    relationships_dedup = relationships_dedup \
        .filter(F.col('source') != F.col('target')) \
        .dropDuplicates(['source', 'target', 'relation'])
    
    # Convert to clean pandas DataFrames (strip Spark metadata to avoid serialization errors)
    print("\n🔄 Converting to pandas (stripping Spark metadata)...")
    entities_df_dedup = pd.DataFrame(entities_dedup.toPandas().to_dict('records'))
    relationships_df_dedup = pd.DataFrame(relationships_dedup.toPandas().to_dict('records'))
    
    print(f"\n📊 After deduplication:")
    print(f"   Entities: {len(entities_df_dedup):,} (reduced by {entity_count - len(entities_df_dedup):,})")
    print(f"   Relationships: {len(relationships_df_dedup):,} (reduced by {rel_count - len(relationships_df_dedup):,})")

# Show top entities by connectivity
if len(relationships_df_dedup) > 0:
    print(f"\n🎯 Top entities by connectivity:")
    entity_counts = defaultdict(int)
    for _, rel in relationships_df_dedup.iterrows():
        entity_counts[rel['source']] += 1
        entity_counts[rel['target']] += 1
    
    top_entities = sorted(entity_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    for entity, count in top_entities:
        print(f"   {entity}: {count} connections")

# Save deduplicated graph
print("\n" + "="*70)
print("💾 Saving deduplicated knowledge graph...")

entities_df_dedup.to_parquet(f"{output_path}/entities.parquet", index=False)
relationships_df_dedup.to_parquet(f"{output_path}/relationships.parquet", index=False)
entities_df_dedup.to_parquet(f"{output_path}/entities_dedup.parquet", index=False)
relationships_df_dedup.to_parquet(f"{output_path}/relationships_dedup.parquet", index=False)

print(f"✅ Saved to {output_path}/")
print("   - entities.parquet ({:,} entities)".format(len(entities_df_dedup)))
print("   - relationships.parquet ({:,} relationships)".format(len(relationships_df_dedup)))
print("⚡ Optimization: Spark parallel similarity matching, type-based grouping")
print("⚡ Serverless: Auto-optimized, no manual caching needed")
print("="*70)

print("\n✅ Deduplication complete! Graph is now cleaner and more powerful.\n")

In [0]:
# Automatic Backup After Job Completion
# Runs after Cell 10 (deduplication) to backup knowledge graph and cleanup old backups
# This cell should be the LAST cell in your scheduled job

from datetime import datetime
import os

print("="*70)
print("💾 AUTOMATIC POST-JOB BACKUP")
print("="*70)

output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"

# Check if output exists
if not os.path.exists(output_path):
    print("\n⚠️  Output directory not found. Skipping backup.")
else:
    # Count files in output
    files = [f for f in os.listdir(output_path) if not f.startswith('_backup')]
    file_count = len(files)
    
    print(f"\n📂 Output directory status:")
    print(f"   Files: {file_count}")
    print(f"   Location: {output_path}")
    
    if file_count == 0:
        print("\n⚠️  Output directory is empty. Skipping backup.")
    else:
        # Create timestamped backup
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
        print(f"\n💾 Creating backup: {timestamp}")
        
        try:
            # Call backup function from Cell 11
            backup_path = backup_output("scheduled-job")
            
            if backup_path:
                print(f"\n✅ Backup successful!")
                print(f"   Backup location: {backup_path}")
                
                # Cleanup old backups (keep last 7 days)
                print(f"\n🧹 Cleaning up old backups...")
                cleanup_old_backups(keep_days=7)
                
                print(f"\n✅ Post-job backup complete!")
            else:
                print(f"\n⚠️  Backup failed. Check backup function output above.")
                
        except NameError:
            print("\n⚠️  Backup functions not found. Make sure Cell 11 has been run.")
            print("   For scheduled jobs, ensure Cell 11 is included before this cell.")
        except Exception as e:
            print(f"\n❌ Backup error: {str(e)}")

print("\n" + "="*70)
print("✅ JOB COMPLETION TASKS FINISHED")
print("="*70)
print(f"\n📊 Job Summary:")
print(f"   Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Output location: {output_path}")
print(f"   Backup: Completed")
print(f"   Old backups cleaned: Completed")
print("\n🚀 Knowledge graph is ready for use!")
print("="*70)

In [0]:
# Backup & Restore Utilities
# Daily backup of knowledge graph output directory
# Administrators can restore from previous day's backup in case of emergency

import os
import shutil
from datetime import datetime, timedelta
import pandas as pd

print("💾 BACKUP & RESTORE UTILITIES")
print("="*70)

output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"
backup_base_path = "/Volumes/research_catalog/healthcare/policy_docs/backups"

# Create backup directory if it doesn't exist
os.makedirs(backup_base_path, exist_ok=True)

# ===================================================================
# BACKUP FUNCTION
# ===================================================================

def backup_output(description=""):
    """
    Create a dated backup of the output directory.
    
    Args:
        description: Optional description for this backup (e.g., "post-daily-run", "before-reprocess")
    
    Returns:
        Backup directory path
    """
    # Generate timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    date_str = datetime.now().strftime("%Y-%m-%d")
    
    # Create backup folder name
    if description:
        backup_folder = f"backup_{timestamp}_{description}"
    else:
        backup_folder = f"backup_{timestamp}"
    
    backup_path = os.path.join(backup_base_path, backup_folder)
    
    print(f"\n📊 Creating backup: {backup_folder}")
    print(f"   Source: {output_path}")
    print(f"   Destination: {backup_path}")
    
    try:
        # Copy entire output directory
        shutil.copytree(output_path, backup_path)
        
        # Count files backed up
        file_count = sum([len(files) for _, _, files in os.walk(backup_path)])
        
        # Get total size
        total_size = 0
        for dirpath, dirnames, filenames in os.walk(backup_path):
            for filename in filenames:
                filepath = os.path.join(dirpath, filename)
                total_size += os.path.getsize(filepath)
        
        size_mb = total_size / (1024 * 1024)
        
        # Save backup metadata
        metadata = {
            'backup_date': date_str,
            'backup_timestamp': timestamp,
            'description': description,
            'file_count': file_count,
            'size_mb': round(size_mb, 2),
            'backup_path': backup_path
        }
        
        metadata_df = pd.DataFrame([metadata])
        metadata_df.to_parquet(os.path.join(backup_path, "_backup_metadata.parquet"), index=False)
        
        print(f"\n✅ Backup complete!")
        print(f"   Files backed up: {file_count}")
        print(f"   Total size: {size_mb:.2f} MB")
        print(f"   Backup location: {backup_path}")
        
        return backup_path
        
    except Exception as e:
        print(f"\n❌ Backup failed: {str(e)}")
        return None

# ===================================================================
# RESTORE FUNCTION
# ===================================================================

def restore_from_backup(backup_name=None):
    """
    Restore output directory from a backup.
    
    Args:
        backup_name: Name of backup folder (e.g., "backup_20260403_140530")
                     If None, restores from most recent backup
    
    Returns:
        True if successful, False otherwise
    """
    # Find backup
    if backup_name is None:
        # Get most recent backup
        backups = [d for d in os.listdir(backup_base_path) if d.startswith('backup_')]
        if not backups:
            print("❌ No backups found!")
            return False
        backups.sort(reverse=True)  # Most recent first
        backup_name = backups[0]
        print(f"ℹ️  No backup specified, using most recent: {backup_name}")
    
    backup_path = os.path.join(backup_base_path, backup_name)
    
    if not os.path.exists(backup_path):
        print(f"❌ Backup not found: {backup_path}")
        return False
    
    print(f"\n⚠️  WARNING: This will replace the current output directory!")
    print(f"   Current: {output_path}")
    print(f"   Restore from: {backup_path}")
    
    # Load backup metadata if available
    metadata_file = os.path.join(backup_path, "_backup_metadata.parquet")
    if os.path.exists(metadata_file):
        metadata = pd.read_parquet(metadata_file)
        print(f"\n📄 Backup Info:")
        print(f"   Date: {metadata['backup_date'].iloc[0]}")
        print(f"   Description: {metadata['description'].iloc[0]}")
        print(f"   Files: {metadata['file_count'].iloc[0]}")
        print(f"   Size: {metadata['size_mb'].iloc[0]} MB")
    
    response = input("\n❓ Proceed with restore? (yes/no): ")
    
    if response.lower() != 'yes':
        print("❌ Restore cancelled")
        return False
    
    try:
        # Backup current state first (safety)
        safety_backup = backup_output("pre-restore-safety")
        print(f"\n🔒 Safety backup created: {os.path.basename(safety_backup)}")
        
        # Remove current output directory
        print(f"\n🗑️  Removing current output directory...")
        shutil.rmtree(output_path)
        
        # Restore from backup
        print(f"🔄 Restoring from backup...")
        shutil.copytree(backup_path, output_path)
        
        # Verify restoration
        file_count = sum([len(files) for _, _, files in os.walk(output_path)])
        
        print(f"\n✅ Restore complete!")
        print(f"   Restored files: {file_count}")
        print(f"   Location: {output_path}")
        
        return True
        
    except Exception as e:
        print(f"\n❌ Restore failed: {str(e)}")
        return False

# ===================================================================
# LIST BACKUPS FUNCTION
# ===================================================================

def list_backups():
    """List all available backups with metadata."""
    backups = [d for d in os.listdir(backup_base_path) if d.startswith('backup_')]
    
    if not backups:
        print("ℹ️  No backups found")
        return
    
    backups.sort(reverse=True)  # Most recent first
    
    print(f"\n📂 Available Backups ({len(backups)} total):\n")
    
    backup_info = []
    for backup_name in backups:
        backup_path = os.path.join(backup_base_path, backup_name)
        metadata_file = os.path.join(backup_path, "_backup_metadata.parquet")
        
        if os.path.exists(metadata_file):
            metadata = pd.read_parquet(metadata_file)
            info = {
                'Backup Name': backup_name,
                'Date': metadata['backup_date'].iloc[0],
                'Description': metadata['description'].iloc[0] or '(none)',
                'Files': metadata['file_count'].iloc[0],
                'Size (MB)': metadata['size_mb'].iloc[0]
            }
        else:
            info = {
                'Backup Name': backup_name,
                'Date': 'Unknown',
                'Description': '(none)',
                'Files': 'Unknown',
                'Size (MB)': 'Unknown'
            }
        
        backup_info.append(info)
    
    backup_df = pd.DataFrame(backup_info)
    display(backup_df)
    
    print(f"\n📍 Backup location: {backup_base_path}")

# ===================================================================
# CLEANUP FUNCTION
# ===================================================================

def cleanup_old_backups(keep_days=7):
    """
    Delete backups older than specified days.
    
    Args:
        keep_days: Number of days to keep backups (default: 7)
    """
    cutoff_date = datetime.now() - timedelta(days=keep_days)
    cutoff_str = cutoff_date.strftime("%Y%m%d")
    
    backups = [d for d in os.listdir(backup_base_path) if d.startswith('backup_')]
    
    deleted_count = 0
    for backup_name in backups:
        # Extract date from backup name (format: backup_YYYYMMDD_HHMMSS)
        try:
            date_part = backup_name.split('_')[1]  # Get YYYYMMDD
            if date_part < cutoff_str:
                backup_path = os.path.join(backup_base_path, backup_name)
                print(f"🗑️  Deleting old backup: {backup_name}")
                shutil.rmtree(backup_path)
                deleted_count += 1
        except:
            pass  # Skip malformed backup names
    
    if deleted_count > 0:
        print(f"\n✅ Cleaned up {deleted_count} old backups (older than {keep_days} days)")
    else:
        print(f"\nℹ️  No backups older than {keep_days} days found")

# ===================================================================
# USAGE INSTRUCTIONS
# ===================================================================

print("\n" + "="*70)
print("📝 USAGE INSTRUCTIONS")
print("="*70)
print("""
🔹 CREATE BACKUP (run after daily pipeline):
   backup_output("daily-run")
   backup_output("before-major-update")

🔹 LIST ALL BACKUPS:
   list_backups()

🔹 RESTORE FROM BACKUP:
   # Restore from most recent
   restore_from_backup()
   
   # Restore from specific backup
   restore_from_backup("backup_20260403_140530_daily-run")

🔹 CLEANUP OLD BACKUPS:
   # Delete backups older than 7 days
   cleanup_old_backups(keep_days=7)
   
   # Delete backups older than 30 days
   cleanup_old_backups(keep_days=30)

🔹 AUTOMATED DAILY BACKUP (add to end of pipeline):
   After Cell 9 completes successfully:
   1. backup_output("daily-run")
   2. cleanup_old_backups(keep_days=7)
""")
print("="*70)

# Show current status
if os.path.exists(output_path):
    files = os.listdir(output_path)
    print(f"\nℹ️  Current output directory: {len(files)} files")
else:
    print(f"\n⚠️  Output directory does not exist: {output_path}")

backups = [d for d in os.listdir(backup_base_path) if d.startswith('backup_')]
print(f"ℹ️  Available backups: {len(backups)}")

if backups:
    backups.sort(reverse=True)
    print(f"ℹ️  Most recent backup: {backups[0]}")

In [0]:
# Graph-Aware Relationship Chatbot
# Uses the knowledge graph (entities + relationships) to answer questions like:
# "How are lung cancer screening and telehealth related?"
# "What policies affect Medicare beneficiaries?"

import pandas as pd
from difflib import SequenceMatcher
from openai import OpenAI
import re

print("🧠 KNOWLEDGE GRAPH CHATBOT")
print("="*70)
print("This chatbot can answer relationship questions using the knowledge graph.")
print("="*70)

# Setup OpenAI client for Databricks
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
client = OpenAI(
    api_key=token,
    base_url=f"https://{workspace_url}/serving-endpoints"
)

# Load knowledge graph and text units
output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"

try:
    entities_df = pd.read_parquet(f"{output_path}/entities.parquet")
    relationships_df = pd.read_parquet(f"{output_path}/relationships.parquet")
    text_units_df = pd.read_parquet(f"{output_path}/text_units.parquet")
    text_col = 'text' if 'text' in text_units_df.columns else text_units_df.columns[0]
    
    print(f"\n✅ Knowledge graph loaded:")
    print(f"   Entities: {len(entities_df)}")
    print(f"   Relationships: {len(relationships_df)}")
    print(f"   Text chunks: {len(text_units_df)}\n")
except FileNotFoundError:
    print("\n⚠️  Knowledge graph not found. Run the extraction cell first!")
    entities_df = pd.DataFrame()
    relationships_df = pd.DataFrame()
    text_units_df = pd.DataFrame()
    text_col = 'text'

# Normalize text for better matching
def normalize_text(text):
    """Normalize text by removing special chars and splitting on hyphens/underscores."""
    # Convert to lowercase
    text = text.lower()
    # Replace hyphens, underscores, slashes with spaces
    text = re.sub(r'[-_/]', ' ', text)
    # Remove punctuation except spaces
    text = re.sub(r'[^\w\s]', ' ', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Find entities mentioned in text
def find_entities_in_query(query, threshold=0.3):
    """Find entities mentioned in the query using improved fuzzy matching."""
    if len(entities_df) == 0:
        return []
    
    query_lower = query.lower()
    query_normalized = normalize_text(query)
    query_words = set(query_normalized.split())
    
    matches = []
    
    for _, entity in entities_df.iterrows():
        entity_text = entity['text']
        entity_lower = entity_text.lower()
        entity_normalized = normalize_text(entity_text)
        entity_words = set(entity_normalized.split())
        
        # 1. Exact substring match (highest priority)
        if entity_lower in query_lower or query_lower in entity_lower:
            matches.append((entity_text, entity['type'], 1.0))
            continue
        
        # 2. Normalized substring match
        if entity_normalized in query_normalized or query_normalized in entity_normalized:
            matches.append((entity_text, entity['type'], 0.95))
            continue
        
        # 3. Word overlap match (improved - uses normalized words)
        overlap = len(entity_words & query_words)
        if overlap > 0:
            # Calculate score based on overlap
            score = overlap / max(len(entity_words), len(query_words))
            if score >= threshold:
                matches.append((entity_text, entity['type'], score))
                continue
        
        # 4. Partial word matching (e.g., "Medicare" matches "medicare-preventive-services")
        partial_matches = 0
        for entity_word in entity_words:
            for query_word in query_words:
                if len(entity_word) > 3 and len(query_word) > 3:  # Skip short words
                    if entity_word in query_word or query_word in entity_word:
                        partial_matches += 1
                        break
        
        if partial_matches > 0:
            score = partial_matches / max(len(entity_words), len(query_words))
            if score >= threshold:
                matches.append((entity_text, entity['type'], score * 0.8))  # Lower weight for partial
    
    # Remove duplicates and sort by match score
    seen = set()
    unique_matches = []
    for match in matches:
        if match[0] not in seen:
            seen.add(match[0])
            unique_matches.append(match)
    
    unique_matches.sort(key=lambda x: x[2], reverse=True)
    return unique_matches[:10]  # Top 10 matches

# Find relationships between entities
def find_relationships(entity_names):
    """Find relationships involving the given entities."""
    if len(relationships_df) == 0:
        return []
    
    entity_names_lower = [e.lower() for e in entity_names]
    
    relevant_rels = []
    for _, rel in relationships_df.iterrows():
        source_lower = rel['source'].lower()
        target_lower = rel['target'].lower()
        
        # Check if any entity is involved
        for entity in entity_names_lower:
            if entity in source_lower or entity in target_lower:
                relevant_rels.append(rel)
                break
    
    return relevant_rels

# Get chunks containing relationships
def get_relationship_chunks(relationships):
    """Get text chunks that contain the relationships."""
    if len(relationships) == 0 or len(text_units_df) == 0:
        return []
    
    chunk_ids = set(rel['chunk_id'] for rel in relationships)
    chunks = []
    
    for chunk_id in chunk_ids:
        if chunk_id < len(text_units_df):
            chunks.append((chunk_id, str(text_units_df.iloc[chunk_id][text_col])))
    
    return chunks

# Main graph-aware question answering
def ask_graph(query, show_details=True):
    """Answer relationship questions using the knowledge graph."""
    print(f"🔍 Analyzing: {query}\n")
    
    # Step 1: Find entities in query
    entities = find_entities_in_query(query)
    
    if not entities:
        return "⚠️  No entities found in your question. Try asking about specific policies, procedures, or organizations mentioned in the documents."
    
    if show_details:
        print("🎯 Entities found:")
        for text, etype, score in entities[:5]:
            print(f"   - {text} ({etype}) [match: {score:.2f}]")
        print()
    
    # Step 2: Find relationships
    entity_names = [e[0] for e in entities]
    relationships = find_relationships(entity_names)
    
    if not relationships:
        return "⚠️  Found entities but no relationships between them in the knowledge graph. Try different terms or ask about specific connections."
    
    if show_details:
        print(f"🔗 Relationships found: {len(relationships)}")
        for rel in relationships[:5]:
            print(f"   {rel['source']} → {rel['relation']} → {rel['target']}")
        print()
    
    # Step 3: Get relevant chunks
    chunks = get_relationship_chunks(relationships)
    
    if show_details:
        print(f"📚 Relevant chunks: {len(chunks)}\n")
    
    # Step 4: Build context
    context_parts = []
    
    # Add relationship information
    context_parts.append("KNOWLEDGE GRAPH RELATIONSHIPS:")
    for rel in relationships[:10]:  # Top 10 relationships
        context_parts.append(f"- {rel['source']} {rel['relation']} {rel['target']}")
    
    context_parts.append("\nRELEVANT TEXT PASSAGES:")
    for chunk_id, chunk_text in chunks[:3]:  # Top 3 chunks
        context_parts.append(chunk_text[:500])  # First 500 chars
    
    context = "\n\n".join(context_parts)
    
    # Step 5: Generate answer with LLM
    if show_details:
        print("🤖 Generating answer...\n")
    
    system_prompt = """You are a Medicare coverage policy expert assistant.
Answer questions about relationships between policies, procedures, organizations, and conditions.
Use both the knowledge graph relationships AND the text passages provided.
Explain how entities are connected and what policies govern them."""
    
    user_prompt = f"""Context:
{context}

Question: {query}

Provide a clear answer explaining the relationships and connections."""
    
    try:
        response = client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=600,
            temperature=0.2
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"❌ LLM Error: {e}")
        return f"Here's what I found:\n\n{context[:1500]}..."

# Simple wrapper for interactive use
def ask_relationship(query):
    """Ask a relationship question - simple wrapper."""
    print("="*70)
    answer = ask_graph(query, show_details=True)  # Show details for demo
    print("📝 Answer:")
    print("="*70)
    print(answer)
    print("="*70)
    return answer

print("\n🚀 READY! Two chatbot modes available:\n")
print("1. 📦 Simple RAG (keyword search):")
print("   ask('What are lung cancer screening requirements?')")
print()
print("2. 🧠 Graph-Aware (relationship questions):")
print("   ask_graph('How are lung cancer screening and Medicare related?')")
print("   ask_relationship('What policies affect Medicare beneficiaries?')")
print()
print("="*70)

# ===================================================================
# TEST RUN (Commented out for scheduled jobs - uncomment for testing)
# ===================================================================
# if len(entities_df) > 0:
#     print("\n🧪 Testing graph chatbot...\n")
#     test_answer = ask_relationship("How are Medicare preventive services and screening procedures related?")

# 📖 Chatbot Usage Guide

## Quick Start
Simply run: `ask('your question')`

## Example Questions by Category

### National Coverage Determinations (NCDs)
```python
ask('What are the NCD requirements for lung cancer screening?')
ask('What NCDs cover preventive services?')
ask('What conditions are covered under Medicare NCDs?')
```

### Local Coverage Determinations (LCDs)
```python
ask('What LCDs apply to cardiovascular screening?')
ask('Which contractors have LCDs for diabetes screening?')
ask('What are the LCD requirements for LDCT screening?')
```

### HCPCS Codes & Coverage
```python
ask('What HCPCS codes are covered for preventive services?')
ask('Which procedures use HCPCS modifiers?')
ask('What HCPCS code groups are included in screening tests?')
```

### ICD-10 Codes & Diagnoses
```python
ask('What ICD-10 codes are covered for cervical cancer screening?')
ask('Which ICD-10 codes are non-covered for specific procedures?')
ask('What diagnoses are covered under preventive care?')
```

### Medicare Part D
```python
ask('What changed in the Part D redesign for 2026?')
ask('What are the Part D benefit requirements?')
ask('How does Medicare Part D affect prescription drug plans?')
```

### Contractors & Jurisdictions
```python
ask('Which contractors cover my state?')
ask('What contractors are responsible for article enforcement?')
ask('What jurisdictions apply to DME regional contractors?')
```

### Coverage Articles
```python
ask('What articles relate to respiratory screening?')
ask('Which articles have future retirement dates?')
ask('What response to comments are available for screening articles?')
```

## Graph-Aware Relationship Questions

Use `ask_relationship()` for questions about how entities are connected:

```python
ask_relationship('How are Medicare Part D and prescription drugs related?')
ask_relationship('What policies affect Medicare beneficiaries?')
ask_relationship('How are NCDs and preventive services connected?')
ask_relationship('What procedures does CMS provide coverage for?')
ask_relationship('How are HCPCS codes and screening tests related?')
```

## Advanced Usage

For more control, use `ask_question()` directly:
```python
# Show source passages
answer = ask_question('your question', show_sources=True)

# Hide source passages for cleaner output  
answer = ask_question('your question', show_sources=False)
```

## What's Happening Behind the Scenes

1. **Keyword Search**: Finds the 3 most relevant text chunks from your document collection
2. **Context Building**: Combines relevant passages as context
3. **LLM Generation**: Uses Llama 3.3 70B to generate accurate answers based on Medicare coverage data

## Data Source: CMS Medicare Coverage Database

- **Source**: https://www.cms.gov/medicare-coverage-database/downloads/downloads.aspx
- **Dataset**: "All Data" download (NCDs, LCDs, Articles, Code mappings)
- **Processing**: 
  - 67 files (61 CSVs + 6 PDFs)
  - Converted to text chunks (~1200 tokens each)
  - HTML tags cleaned from CSV content
- **Graph Structure**:
  - 241 entities (policies, procedures, organizations, conditions, demographics)
  - 311 relationships (includes, affects, provides, covers, requires)
- **Location**: `/Volumes/research_catalog/healthcare/policy_docs/`

## Entity Types in Graph

- **Policies**: NCDs, LCDs, Medicare programs, Part D plans
- **Procedures**: LDCT, screening tests, DSMT services, medical procedures
- **Organizations**: CMS, Medicare contractors, plan sponsors, agencies
- **Conditions**: Cancer, cardiovascular disease, diabetes, other medical conditions
- **Demographics**: Medicare beneficiaries, age groups, patient categories

---

💡 **Tip**: This chatbot contains the complete CMS Medicare Coverage Database including all NCDs, LCDs, Articles, and code mappings. For the most accurate answers, reference specific policy numbers, HCPCS codes, or ICD-10 codes in your questions.

In [0]:
# 💬 Test Simple Chatbot
# Use the ask() function to test the keyword-based RAG chatbot.
# This chatbot searches text chunks using keyword matching.

# Example questions based on CMS Medicare Coverage Database:

# National Coverage Determinations
ask('What are the lung cancer screening NCD requirements?')

# Uncomment to try more questions:

# HCPCS Codes
# ask('What HCPCS codes are covered for preventive services?')
# ask('Which HCPCS modifiers are used for screening procedures?')

# ICD-10 Codes
# ask('What ICD-10 codes are covered for cardiovascular screening?')
# ask('Which diagnoses are non-covered for specific procedures?')

# Medicare Part D
# ask('What changed in Medicare Part D redesign for 2026?')
# ask('What are the Part D benefit requirements for prescription drugs?')

# Contractors & Jurisdictions
# ask('Which contractors are responsible for LCD enforcement?')
# ask('What jurisdictions apply to my state for Medicare coverage?')

# Coverage Articles
# ask('What articles cover respiratory screening procedures?')
# ask('Which local coverage articles have future retirement dates?')

In [0]:
# 🧠 Test Graph Chatbot
# Use ask_graph() or ask_relationship() to test the graph-aware chatbot.
# This chatbot answers questions about relationships between entities.

# Example relationship questions based on actual graph entities:

# Policy-Procedure Relationships
# ask_relationship('How are Medicare preventive services and screening procedures related?')

# Uncomment to try more relationship questions:

# Organization-Policy Relationships
# ask_graph('What policies does CMS provide?')
# ask_relationship('How is CMS related to Medicare Part D?')

# Policy-Demographics Relationships
# ask_relationship('What policies affect Medicare beneficiaries?')
# ask_graph('How are preventive services and Medicare beneficiaries connected?')

# Procedure-Condition Relationships
# ask_relationship('How are LDCT screening and lung cancer related?')
# ask_graph('What procedures cover cardiovascular conditions?')

# Policy-Code Relationships
# ask_relationship('What HCPCS codes are included in preventive services?')
# ask_graph('How are ICD-10 codes related to screening procedures?')

# Complex Multi-Entity Questions
# ask_relationship('How are NCDs, LCDs, and contractors connected?')
# ask_graph('What is the relationship between Part D plans and prescription drugs?')
# ask_relationship('How do screening tests relate to coverage determinations?')

ask_relationship('What policies affect Medicare beneficiaries?')

In [0]:
# DATA SYNC & CONSISTENCY VALIDATION
# Comprehensive checks to verify job completed successfully and all data is consistent.
# Run this after job completion to validate:
#   1. All input documents were processed
#   2. Output artifacts exist and are valid
#   3. Data consistency across pipeline stages
#   4. Coverage and quality metrics

import os
import pandas as pd
from collections import defaultdict
from datetime import datetime

print("="*80)
print("🔍 DATA SYNC & CONSISTENCY VALIDATION")
print("="*80)
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

input_path = "/Volumes/research_catalog/healthcare/policy_docs/input"
output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"

# Track validation results
validation_results = {
    'passed': [],
    'warnings': [],
    'errors': []
}

def log_pass(message):
    validation_results['passed'].append(message)
    print(f"✅ {message}")

def log_warning(message):
    validation_results['warnings'].append(message)
    print(f"⚠️  {message}")

def log_error(message):
    validation_results['errors'].append(message)
    print(f"❌ {message}")

# ========================================
# 1. INPUT DIRECTORY VALIDATION
# ========================================
print("\n" + "="*80)
print("📂 STEP 1: INPUT DIRECTORY VALIDATION")
print("="*80)

if not os.path.exists(input_path):
    log_error(f"Input directory not found: {input_path}")
else:
    log_pass(f"Input directory exists: {input_path}")
    
    # Count files by type
    input_files = os.listdir(input_path)
    txt_files = [f for f in input_files if f.endswith('.txt')]
    pdf_files = [f for f in input_files if f.endswith('.pdf')]
    csv_files = [f for f in input_files if f.endswith('.csv')]
    
    print(f"\n📊 Input File Counts:")
    print(f"   TXT files: {len(txt_files)}")
    print(f"   PDF files: {len(pdf_files)}")
    print(f"   CSV files: {len(csv_files)}")
    print(f"   Total files: {len(input_files)}")
    
    if len(txt_files) == 0:
        log_error("No .txt files found in input directory")
    else:
        log_pass(f"Found {len(txt_files)} text files for processing")
    
    # Check for empty files
    empty_files = []
    for txt_file in txt_files:
        file_path = os.path.join(input_path, txt_file)
        if os.path.getsize(file_path) == 0:
            empty_files.append(txt_file)
    
    if empty_files:
        log_warning(f"Found {len(empty_files)} empty text files: {', '.join(empty_files[:5])}")
    else:
        log_pass("All text files have content")

# ========================================
# 2. OUTPUT DIRECTORY VALIDATION
# ========================================
print("\n" + "="*80)
print("📂 STEP 2: OUTPUT DIRECTORY VALIDATION")
print("="*80)

required_files = [
    'text_units.parquet',
    'entities.parquet',
    'relationships.parquet',
    'entities_dedup.parquet',
    'relationships_dedup.parquet'
]

if not os.path.exists(output_path):
    log_error(f"Output directory not found: {output_path}")
else:
    log_pass(f"Output directory exists: {output_path}")
    
    print(f"\n📊 Required Output Files:")
    for file_name in required_files:
        file_path = os.path.join(output_path, file_name)
        if os.path.exists(file_path):
            file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f"   ✅ {file_name:30s} ({file_size_mb:.2f} MB)")
            
            if file_size_mb < 0.001:
                log_warning(f"{file_name} exists but is nearly empty ({file_size_mb:.4f} MB)")
            else:
                log_pass(f"{file_name} exists with valid size")
        else:
            print(f"   ❌ {file_name:30s} (MISSING)")
            log_error(f"Required file missing: {file_name}")

# ========================================
# 3. DATA CONSISTENCY CHECKS
# ========================================
print("\n" + "="*80)
print("🔍 STEP 3: DATA CONSISTENCY CHECKS")
print("="*80)

try:
    # Load data files
    text_units_df = pd.read_parquet(os.path.join(output_path, 'text_units.parquet'))
    entities_df = pd.read_parquet(os.path.join(output_path, 'entities.parquet'))
    relationships_df = pd.read_parquet(os.path.join(output_path, 'relationships.parquet'))
    
    print(f"\n📊 Data Loaded Successfully:")
    print(f"   Text Units: {len(text_units_df):,} rows")
    print(f"   Entities: {len(entities_df):,} rows")
    print(f"   Relationships: {len(relationships_df):,} rows")
    
    # Check 3.1: Document Coverage
    print(f"\n🔍 Check 3.1: Document Coverage")
    input_txt_files = set(txt_files)
    processed_docs = set(text_units_df['document_id'].unique())
    
    missing_docs = input_txt_files - processed_docs
    if missing_docs:
        log_warning(f"Found {len(missing_docs)} unprocessed documents: {', '.join(list(missing_docs)[:5])}")
    else:
        log_pass(f"All {len(input_txt_files)} input documents were chunked")
    
    extra_docs = processed_docs - input_txt_files
    if extra_docs:
        log_warning(f"Found {len(extra_docs)} documents in output not in input (may be from previous runs)")
    
    # Check 3.2: Chunk Coverage
    print(f"\n🔍 Check 3.2: Chunk Coverage")
    total_chunks = len(text_units_df)
    if total_chunks == 0:
        log_error("No text chunks found in text_units.parquet")
    else:
        log_pass(f"Generated {total_chunks:,} text chunks from documents")
        
        # Check for duplicate chunk IDs
        duplicate_ids = text_units_df[text_units_df.duplicated(subset=['id'], keep=False)]
        if len(duplicate_ids) > 0:
            log_warning(f"Found {len(duplicate_ids)} duplicate chunk IDs")
        else:
            log_pass("All chunk IDs are unique")
    
    # Check 3.3: Entity Extraction Completion (FIXED METRIC)
    print(f"\n🔍 Check 3.3: Entity Extraction Completion")
    if len(entities_df) == 0:
        log_error("No entities found - extraction may have failed")
    else:
        # Check for checkpoint files (indicates incomplete extraction)
        checkpoint_files = [f for f in os.listdir(output_path) if f.startswith('entities_checkpoint_')]
        
        if checkpoint_files:
            log_warning(f"Found {len(checkpoint_files)} checkpoint files - extraction may be incomplete")
            print(f"   Run Cell 9 again to complete extraction, or manually delete checkpoints if complete")
        else:
            log_pass(f"No checkpoint files found - extraction completed successfully")
        
        # Additional metric: entities per chunk
        chunks_with_entities = entities_df['chunk_id'].nunique()
        entities_per_chunk = len(entities_df) / total_chunks
        print(f"   Chunks with entities: {chunks_with_entities:,} / {total_chunks:,} ({chunks_with_entities/total_chunks*100:.1f}%)")
        print(f"   Average entities per chunk: {entities_per_chunk:.2f}")
        
        if entities_per_chunk < 0.1:
            log_warning(f"Low entity density: {entities_per_chunk:.2f} entities/chunk (may indicate extraction issues)")
        else:
            log_pass(f"Reasonable entity density: {entities_per_chunk:.2f} entities/chunk")
    
    # Check 3.4: Entity Types Distribution
    print(f"\n🔍 Check 3.4: Entity Types")
    if 'type' in entities_df.columns:
        entity_types = entities_df['type'].value_counts()
        print(f"   Found {len(entity_types)} entity types:")
        for entity_type, count in entity_types.items():
            print(f"      • {entity_type:20s}: {count:>6,}")
        log_pass(f"Extracted {len(entity_types)} different entity types")
    else:
        log_warning("Entity 'type' column not found")
    
    # Check 3.5: Relationship Validity
    print(f"\n🔍 Check 3.5: Relationship Validity")
    if len(relationships_df) == 0:
        log_warning("No relationships found - may indicate entity extraction issues")
    else:
        log_pass(f"Found {len(relationships_df):,} relationships between entities")
        
        # Check for orphaned relationships (references to non-existent entities)
        entity_texts = set(entities_df['text'].unique())
        sources = set(relationships_df['source'].unique())
        targets = set(relationships_df['target'].unique())
        
        orphaned_sources = sources - entity_texts
        orphaned_targets = targets - entity_texts
        
        if orphaned_sources:
            log_warning(f"Found {len(orphaned_sources)} relationship sources not in entities table")
        else:
            log_pass("All relationship sources reference valid entities")
        
        if orphaned_targets:
            log_warning(f"Found {len(orphaned_targets)} relationship targets not in entities table")
        else:
            log_pass("All relationship targets reference valid entities")
        
        # Check for self-loops
        self_loops = relationships_df[relationships_df['source'] == relationships_df['target']]
        if len(self_loops) > 0:
            log_warning(f"Found {len(self_loops)} self-loop relationships (entity→itself)")
        else:
            log_pass("No self-loop relationships found")
    
    # Check 3.6: Data Quality
    print(f"\n🔍 Check 3.6: Data Quality")
    
    # Check for null values in critical columns
    null_checks = [
        ('text_units', 'text', text_units_df['text'].isnull().sum()),
        ('entities', 'text', entities_df['text'].isnull().sum()),
        ('relationships', 'source', relationships_df['source'].isnull().sum()),
        ('relationships', 'target', relationships_df['target'].isnull().sum()),
    ]
    
    for table, column, null_count in null_checks:
        if null_count > 0:
            log_warning(f"Found {null_count} null values in {table}.{column}")
        else:
            log_pass(f"No null values in {table}.{column}")
    
    # Check for empty text values
    empty_texts = (text_units_df['text'].str.strip() == '').sum()
    if empty_texts > 0:
        log_warning(f"Found {empty_texts} empty text chunks")
    else:
        log_pass("All text chunks have content")
    
except FileNotFoundError as e:
    log_error(f"Could not load data files: {str(e)}")
except Exception as e:
    log_error(f"Error during consistency checks: {str(e)}")

# ========================================
# 4. FINAL SUMMARY
# ========================================
print("\n" + "="*80)
print("📊 VALIDATION SUMMARY")
print("="*80)

print(f"\n✅ Passed Checks: {len(validation_results['passed'])}")
print(f"⚠️  Warnings: {len(validation_results['warnings'])}")
print(f"❌ Errors: {len(validation_results['errors'])}")

if validation_results['errors']:
    print(f"\n❌ CRITICAL ERRORS FOUND:")
    for error in validation_results['errors']:
        print(f"   • {error}")
    print(f"\n⚠️  ACTION REQUIRED: Review errors above and re-run failed pipeline stages")
    final_status = "FAILED"
elif validation_results['warnings']:
    print(f"\n⚠️  WARNINGS FOUND:")
    for warning in validation_results['warnings']:
        print(f"   • {warning}")
    print(f"\n✅ Job completed but review warnings above")
    final_status = "COMPLETED_WITH_WARNINGS"
else:
    print(f"\n✅ ALL CHECKS PASSED")
    print(f"\n🎉 Job completed successfully with no issues!")
    final_status = "SUCCESS"

print("\n" + "="*80)
print(f"FINAL STATUS: {final_status}")
print("="*80)

# Return status for programmatic access
final_status